In [1]:
import pandas as pd

In [2]:
#from nsepy import get_history as gh
import datetime as dt

In [3]:
start = dt.datetime(2013,6,1)
end = dt.datetime(2022,2,11)
#stk_data = gh(symbol='TATACOFFEE',start=start,end=end)
stk_data = pd.read_csv("Tatacoffee13_21.csv", index_col='Date', parse_dates=True)


In [4]:
#stk_data=stk_data[["Open","High","Low","Close"]]
#stk_data.to_csv("Tatacoffee13_21.csv")
stk_data=stk_data.loc[start:end]

In [ ]:
#column="Close"

In [5]:
from sklearn.preprocessing import MinMaxScaler
Ms = MinMaxScaler()
data1= Ms.fit_transform(stk_data)
print("Len:",data1.shape)

Len: (2120, 4)


In [6]:
data1

array([[1.        , 0.99211302, 0.99690908, 1.        ],
       [0.97770967, 1.        , 0.9983302 , 0.9962976 ],
       [0.97492338, 0.97325286, 0.99548797, 0.99442877],
       ...,
       [0.10762051, 0.10493107, 0.10754254, 0.10691114],
       [0.10490387, 0.11734449, 0.10516218, 0.11914669],
       [0.11859153, 0.11628146, 0.11621132, 0.11491537]], shape=(2120, 4))

In [7]:
data1=pd.DataFrame(data1,columns=["Open","High","Low","Close"])

In [8]:
data1

,Open,High,Low,Close
0,1.000000,0.992113,0.996909,1.000000
1,0.977710,1.000000,0.998330,0.996298
2,0.974923,0.973253,0.995488,0.994429
3,0.986069,0.973253,1.000000,0.997391
4,0.986904,0.983437,0.993356,0.988928
...,...,...,...,...
2115,0.106924,0.107606,0.108964,0.109732
2116,0.109083,0.106954,0.109461,0.108286
2117,0.107621,0.104931,0.107543,0.106911
2118,0.104904,0.117344,0.105162,0.119147


In [9]:
import warnings
warnings.filterwarnings("ignore")

In [11]:
# initiating dictionary variable
performance={"Model":[],"RMSE":[],"MaPe":[],"Lag":[],"Test":[]}

In [12]:
# Creating a list of columns and assign to a variable
listt=["Close","High","Open","Low"]

In [16]:
def cominbation(dataset,listt):
    print(listt)
    datasetTwo=dataset[listt]  # converting the order of the column in the dataset
    test_obs = 28  # assigning test observation values in a variable
    train =datasetTwo[:-test_obs]  #Capturing training records upto 28 records from end
    test = datasetTwo[-test_obs:]  # capturing last 28 records and assign to a varaible test
    from statsmodels.tsa.api import VAR   # importing VAR algorithm from statsmodel.tsa.api library
    # Manual lag testing loop which is not required
    #for i in [1,2,3,4,5,6,7,8,9,10]:
    #    model = VAR(train)
    #    results = model.fit(i)
    #    print('Order =', i)
    #    print('AIC: ', results.aic)
    #    print('BIC: ', results.bic)
    #    print()
    model = VAR(train)
    x = model.select_order(maxlags=12)  # this statement is testing lag from 1 to 12 and computes aic, bic, hqic,fpe and suggest the best lag for each metric
    order=x.selected_orders["aic"] # based on best metric the best lag has been assigned to a variable order
    result = model.fit(order) # Train the model with suggested best lag
    #result.summary()
    lagged_Values = train.values[-order:]  # Model needs an end point to start forcast (last p observation to start forecast)
    pred = result.forecast(y=lagged_Values,steps=28) # predicting forecast value from the observation point for next n(28) rows
    preds=pd.DataFrame(pred,columns=listt) # Assigning column name to the predicted output
    preds.to_csv("varforecasted_{}.csv".format(test_obs))  # Save the output to external file
    from sklearn.metrics import root_mean_squared_error  # import rmse algorithm from metrics library
    rmse= round(root_mean_squared_error(test,pred)) #calculate rmse value
    from sklearn.metrics import mean_absolute_percentage_error #import mape algorithm from metrics library
    mape=mean_absolute_percentage_error(test,pred) #calculate maps value
    performance["Model"].append(listt)
    performance["RMSE"].append(rmse)
    performance["MaPe"].append(mape)
    performance["Lag"].append(order)
    performance["Test"].append(test_obs)
    perf=pd.DataFrame(performance)
    return perf,result,pred

In [17]:
listt=["Close","High","Open","Low"]
#listt=["AQI_calculated","PM10","PM2.5","NOx","NO2","NO","NH3","SO2","CO",'year']


In [18]:
perf,result,pred=cominbation(data1,listt)

['Close', 'High', 'Open', 'Low']


In [19]:
perf

,Model,RMSE,MaPe,Lag,Test
0,"[Close, High, Open, Low]",0,0.086859,12,28


In [20]:
data1

,Open,High,Low,Close
0,1.000000,0.992113,0.996909,1.000000
1,0.977710,1.000000,0.998330,0.996298
2,0.974923,0.973253,0.995488,0.994429
3,0.986069,0.973253,1.000000,0.997391
4,0.986904,0.983437,0.993356,0.988928
...,...,...,...,...
2115,0.106924,0.107606,0.108964,0.109732
2116,0.109083,0.106954,0.109461,0.108286
2117,0.107621,0.104931,0.107543,0.106911
2118,0.104904,0.117344,0.105162,0.119147
